# ResQVision — Data Collection

---

## Imports

In [1]:
import json
import logging
import os
import re
import shutil
import subprocess
import tarfile
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from tqdm.auto import tqdm

/home/prosper/miniforge3/envs/resqvision-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
NOTEBOOK_DIR   = Path().resolve()
SERVICE_DIR    = NOTEBOOK_DIR.parent
DATA_DIR       = SERVICE_DIR / 'data'

RAW_DIR        = DATA_DIR / 'raw'
RAW_TRAIN_DIR  = RAW_DIR  / 'train'
IMAGES_DIR     = RAW_TRAIN_DIR / 'images'
TARGETS_DIR    = RAW_TRAIN_DIR / 'targets'
LABELS_DIR     = RAW_TRAIN_DIR / 'labels'

METADATA_PATH = DATA_DIR / 'meta-data.json'

PROCESSED_DIR       = DATA_DIR / 'processed'
PROCESSED_TRAIN_DIR = PROCESSED_DIR / 'train'

DOWNLOAD_URLS: Dict[str, str] = {
    'training_data': '',
    'metadata':      ''
}

DAMAGE_CATEGORIES: Dict[int, str] = {
    0: 'no-damage',
    1: 'minor-damage',
    2: 'major-damage',
    3: 'destroyed',
    4: 'un-classified',
}

LOG_DIR  = DATA_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / 'data_collection.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8'),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger('resqvision.ingestion')

logger.info('Configuration loaded.')
logger.info('DATA_DIR    : %s', DATA_DIR)
logger.info('RAW_TRAIN   : %s', RAW_TRAIN_DIR)
logger.info('PROCESSED   : %s', PROCESSED_TRAIN_DIR)
logger.info('LOG_FILE    : %s', LOG_FILE)

2026-04-25 22:41:21  INFO      Configuration loaded.
2026-04-25 22:41:21  INFO      DATA_DIR    : /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data
2026-04-25 22:41:21  INFO      RAW_TRAIN   : /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/raw/train
2026-04-25 22:41:21  INFO      PROCESSED   : /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/processed/train
2026-04-25 22:41:21  INFO      LOG_FILE    : /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/logs/data_collection.log


## Utility / Helper Functions

### Filename parser

In [3]:
_FNAME_RE = re.compile(
    r'^(?P<disaster>[a-z0-9][a-z0-9\-]+)_(?P<uid>\d{8})'
    r'_(?P<phase>pre|post)_disaster(?P<suffix>_target)?\.(?P<ext>[a-z]+)$'
)


def parse_filename(filename: str) -> Optional[Dict[str, str]]:
    m = _FNAME_RE.match(filename)
    if m is None:
        return None
    return {
        'disaster':  m.group('disaster'),
        'uid':       m.group('uid'),
        'phase':     m.group('phase'),
        'is_target': m.group('suffix') is not None,
        'ext':       m.group('ext'),
    }

### Download helpers

In [4]:
def _supports_aria2c() -> bool:
    return shutil.which('aria2c') is not None


def download_file(
    url: str,
    dest: Path,
    *,
    chunk_size: int = 8 * 1024 * 1024,
    use_aria2c: bool = True,
) -> Path:
    if not url:
        raise RuntimeError('Download URL is empty.')
    dest.parent.mkdir(parents=True, exist_ok=True)

    if use_aria2c and _supports_aria2c():
        logger.info('Using aria2c: %s -> %s', url, dest)
        cmd = [
            'aria2c', '--continue=true',
            '--max-connection-per-server=4', '--split=4',
            f'--dir={dest.parent}', f'--out={dest.name}', url,
        ]
        result = subprocess.run(cmd, check=False)
        if result.returncode != 0:
            raise RuntimeError(f'aria2c failed with code {result.returncode}')
        return dest

    logger.info('Using requests: %s -> %s', url, dest)
    headers, resume_pos = {}, 0
    if dest.exists():
        resume_pos = dest.stat().st_size
        headers['Range'] = f'bytes={resume_pos}-'

    with requests.get(url, headers=headers, stream=True, timeout=60) as resp:
        if resp.status_code == 416:
            return dest
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0)) + resume_pos
        mode  = 'ab' if resume_pos else 'wb'
        with open(dest, mode) as fh, tqdm(
            total=total, initial=resume_pos,
            unit='B', unit_scale=True, desc=dest.name,
        ) as pbar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                fh.write(chunk)
                pbar.update(len(chunk))
    return dest

### Archive extraction

In [5]:
def extract_archive(archive_path: Path, dest_dir: Path) -> None:
    logger.info('Extracting %s -> %s', archive_path, dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as tf:
        members = tf.getmembers()
        for member in tqdm(members, desc=f'Extracting {archive_path.name}', unit='file'):
            tf.extract(member, path=dest_dir)
    logger.info('Extraction complete: %d files.', len(members))

### Metadata helpers

In [6]:
def load_metadata(p: Path) -> Tuple[Dict, Path]:
    if not p.exists():
        raise FileNotFoundError(f'Metadata file not found: {p}')
    logger.info('Loading metadata: %s', p)
    with open(p, encoding='utf-8') as fh:
        data = json.load(fh)
    logger.info('Metadata loaded: %d entries.', len(data))
    return data, p

def parse_metadata_entry(filename: str, entry: List) -> Dict[str, Any]:
    geotransform, wkt = entry[0], entry[1]
    parsed = parse_filename(filename)
    return {
        'filename':     filename,
        'disaster':     parsed['disaster']  if parsed else None,
        'uid':          parsed['uid']       if parsed else None,
        'phase':        parsed['phase']     if parsed else None,
        'lon_origin':   geotransform[0],
        'lat_origin':   geotransform[3],
        'pixel_width':  geotransform[1],
        'pixel_height': geotransform[5],
        'crs_wkt':      wkt,
    }

### Label JSON helpers

In [ ]:
_DAMAGE_MAP: Dict[str, str] = {
    'no-damage':     'no-damage',
    'minor-damage':  'minor-damage',
    'major-damage':  'major-damage',
    'destroyed':     'destroyed',
    'un-classified': 'un-classified',
}


def parse_label_json(label_path: Path) -> List[Dict[str, Any]]:
    if not label_path.exists():
        logger.warning('Label file missing: %s', label_path)
        return []
    try:
        with open(label_path, encoding='utf-8') as fh:
            data = json.load(fh)
    except json.JSONDecodeError as exc:
        logger.error('Corrupted label JSON [%s]: %s', label_path, exc)
        return []

    features_geo = data.get('features', {}).get('lng_lat', [])
    features_xy  = data.get('features', {}).get('xy', [])
    if not features_geo:
        return []

    records: List[Dict[str, Any]] = []
    for idx, feat in enumerate(features_geo):
        props  = feat.get('properties', {})
        damage = _DAMAGE_MAP.get(props.get('subtype', ''), 'un-classified')
        xy_wkt = features_xy[idx].get('wkt') if idx < len(features_xy) else None
        records.append({
            'uid':          props.get('uid', ''),
            'feature_type': props.get('feature_type', 'building'),
            'damage_type':  damage,
            'lng_lat_wkt':  feat.get('wkt'),
            'xy_wkt':       xy_wkt,
        })
    return records

### Misc helpers

In [8]:
def list_files(directory: Path, glob: str = '*') -> List[Path]:
    return sorted(directory.glob(glob)) if directory.exists() else []


def file_count(directory: Path, glob: str = '*') -> int:
    return len(list_files(directory, glob))


logger.info('Helper functions ready.')

2026-04-25 22:41:21  INFO      Helper functions ready.


## Data Availability Check & Download

### Check required directories

In [9]:
required_dirs = {
    'images':  IMAGES_DIR,
    'targets': TARGETS_DIR,
    'labels':  LABELS_DIR,
}

missing_dirs = {k: v for k, v in required_dirs.items() if not v.exists()}
present_dirs = {k: v for k, v in required_dirs.items() if v.exists()}

logger.info('=' * 60)
logger.info('DATA AVAILABILITY CHECK')
logger.info('=' * 60)

for name, path in present_dirs.items():
    n = file_count(path)
    logger.info('  OK  %-10s -> %s  (%d files)', name, path, n)

for name, path in missing_dirs.items():
    logger.warning('  MISSING  %-10s -> %s', name, path)

DATA_AVAILABLE = len(missing_dirs) == 0
logger.info('Data complete: %s', DATA_AVAILABLE)
print(f'Data available: {DATA_AVAILABLE}')

2026-04-25 22:41:21  INFO      ============================================================
2026-04-25 22:41:21  INFO      DATA AVAILABILITY CHECK
2026-04-25 22:41:21  INFO      ============================================================
2026-04-25 22:41:22  INFO        OK  images     -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/raw/train/images  (5598 files)
2026-04-25 22:41:22  INFO        OK  targets    -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/raw/train/targets  (5598 files)
2026-04-25 22:41:22  INFO        OK  labels     -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/raw/train/labels  (5598 files)
2026-04-25 22:41:22  INFO      Data complete: True


Data available: True


### Download missing components

> Only runs if `DATA_AVAILABLE == False`. Set URLs in `DOWNLOAD_URLS` to enable.

In [10]:
if DATA_AVAILABLE:
    logger.info('All directories present - download skipped.')
else:
    RAW_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    ARCHIVES_DIR = RAW_DIR / '_archives'
    ARCHIVES_DIR.mkdir(parents=True, exist_ok=True)

    train_url = DOWNLOAD_URLS.get('training_data', '')
    if not train_url:
        logger.warning('No URL set for training_data. Set DOWNLOAD_URLS["training_data"].')
    else:
        archive_path = ARCHIVES_DIR / 'training_data.tar.gz'
        try:
            download_file(train_url, archive_path)
            extract_archive(archive_path, RAW_TRAIN_DIR)
        except Exception as exc:
            logger.error('Training data download/extraction failed: %s', exc)

    if not METADATA_PATH.exists():
        meta_url = DOWNLOAD_URLS.get('metadata', '')
        if not meta_url:
            logger.warning('No URL set for metadata. Set DOWNLOAD_URLS["metadata"].')
        else:
            try:
                download_file(meta_url, METADATA_PATH)
            except Exception as exc:
                logger.error('Metadata download failed: %s', exc)

2026-04-25 22:41:22  INFO      All directories present - download skipped.


### Post-check validation

In [11]:
_counts = {k: file_count(v) for k, v in required_dirs.items()}
n_images, n_targets, n_labels = (
    _counts['images'], _counts['targets'], _counts['labels']
)

for k, v in _counts.items():
    if v == 0:
        logger.warning('WARN: %s/ is empty.', k)
    else:
        logger.info('  %s: %d files', k, v)

print(f'images : {n_images}, targets : {n_targets}, labels : {n_labels}')

2026-04-25 22:41:22  INFO        images: 5598 files
2026-04-25 22:41:22  INFO        targets: 5598 files
2026-04-25 22:41:22  INFO        labels: 5598 files


images : 5598, targets : 5598, labels : 5598


## Metadata Parsing

### Load metadata JSON

In [12]:
raw_metadata, METADATA_AVAILABLE = {}, False
try:
    raw_metadata, _loaded_path = load_metadata(METADATA_PATH)
    METADATA_AVAILABLE = True
except FileNotFoundError:
    logger.error('File not found')

if not METADATA_AVAILABLE:
    logger.error('Metadata not found in any candidate path.')

print(f'Metadata available : {METADATA_AVAILABLE}')
print(f'Total entries      : {len(raw_metadata):,}')

2026-04-25 22:41:22  INFO      Loading metadata: /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/meta-data.json
2026-04-25 22:41:22  INFO      Metadata loaded: 22068 entries.


Metadata available : True
Total entries      : 22,068


### Parse into DataFrame

In [13]:
metadata_records: List[Dict[str, Any]] = []
for fname, entry in tqdm(raw_metadata.items(), desc='Parsing metadata'):
    try:
        metadata_records.append(parse_metadata_entry(fname, entry))
    except Exception as exc:
        logger.warning('Skip metadata entry %s: %s', fname, exc)

df_metadata = pd.DataFrame(metadata_records)
logger.info('Metadata DataFrame: %s', df_metadata.shape)
print(f'Shape: {df_metadata.shape}')
df_metadata.head()

Parsing metadata: 100%|██████████| 22068/22068 [00:00<00:00, 323699.40it/s]
2026-04-25 22:41:22  INFO      Metadata DataFrame: (22068, 9)


Shape: (22068, 9)


,filename,disaster,uid,phase,lon_origin,lat_origin,pixel_width,pixel_height,crs_wkt
0,guatemala-volcano_00000020_pre_disaster.png,guatemala-volcano,00000020,pre,-90.823070,14.414360,0.000004,-0.000004,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS..."
1,guatemala-volcano_00000020_post_disaster.png,guatemala-volcano,00000020,post,-90.823070,14.414360,0.000004,-0.000004,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS..."
2,guatemala-volcano_00000022_pre_disaster.png,guatemala-volcano,00000022,pre,-90.846567,14.391884,0.000004,-0.000004,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS..."
3,guatemala-volcano_00000022_post_disaster.png,guatemala-volcano,00000022,post,-90.846567,14.391884,0.000004,-0.000004,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS..."
4,guatemala-volcano_00000012_pre_disaster.png,guatemala-volcano,00000012,pre,-90.808858,14.364308,0.000004,-0.000004,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS..."


### Summary statistics

In [14]:
if not df_metadata.empty:
    print('-- Disaster types --')
    print(df_metadata['disaster'].value_counts().to_string())
    print('\n-- Phase distribution --')
    print(df_metadata['phase'].value_counts().to_string())

    n_pairs = (
        df_metadata.dropna(subset=['disaster', 'uid'])
        .drop_duplicates(subset=['disaster', 'uid'])
        .shape[0]
    )
    print(f'\nUnique scene pairs: {n_pairs:,}')

    meta_out = PROCESSED_DIR / 'metadata_parsed.csv'
    meta_out.parent.mkdir(parents=True, exist_ok=True)
    df_metadata.to_csv(meta_out, index=False)
    print(f'Saved -> {meta_out}')

-- Disaster types --
disaster
portugal-wildfire      3738
pinery-bushfire        3690
socal-fire             2806
woolsey-fire           1756
nepal-flooding         1238
hurricane-michael      1100
hurricane-florence     1092
hurricane-harvey       1044
midwest-flooding        890
hurricane-matthew       810
santa-rosa-wildfire     754
tuscaloosa-tornado      686
lower-puna-volcano      582
moore-tornado           454
palu-tsunami            392
mexico-earthquake       386
joplin-tornado          298
sunda-tsunami           296
guatemala-volcano        56

-- Phase distribution --
phase
pre     11034
post    11034

Unique scene pairs: 11,034
Saved -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/processed/metadata_parsed.csv


## File Pairing

### Scan and index all raw files

In [15]:
SampleKey = Tuple[str, str]  # (disaster, uid)
samples: Dict[SampleKey, Dict[str, Optional[Path]]] = defaultdict(
    lambda: {'pre': None, 'post': None, 'target': None, 'label': None}
)
unrecognised_files: List[Path] = []

def _ingest_dir(directory: Path, glob_pattern: str, role: str) -> None:
    for path in list_files(directory, glob_pattern):
        info = parse_filename(path.name)
        if info is None:
            unrecognised_files.append(path)
            logger.warning('Unrecognised: %s', path.name)
            continue
        key: SampleKey = (info['disaster'], info['uid'])
        slot = info['phase'] if role == 'image' else role
        if samples[key][slot] is not None:
            logger.warning('Duplicate (%s, %s) slot=%s', *key, slot)
        samples[key][slot] = path

logger.info('Scanning images ...')
_ingest_dir(IMAGES_DIR,  '*.png',  'image')
logger.info('Scanning targets ...')
_ingest_dir(TARGETS_DIR, '*.png',  'target')
logger.info('Scanning labels ...')
_ingest_dir(LABELS_DIR,  '*.json', 'label')

print(f'Unique scene pairs: {len(samples):,}')
print(f'Unrecognised files: {len(unrecognised_files)}')

2026-04-25 22:41:22  INFO      Scanning images ...
2026-04-25 22:41:23  INFO      Scanning targets ...
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000000) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000001) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000002) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000006) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000007) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000008) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000010) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000013) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000015) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 00000016) slot=target
2026-04-25 22:41:23  WARNING   Duplicate (guatemala-volcano, 000000

Unique scene pairs: 2,799
Unrecognised files: 0


### Validate pairs

In [16]:
REQUIRED_SLOTS = {'pre', 'post', 'target', 'label'}
complete_samples:   Dict[SampleKey, Dict] = {}
incomplete_samples: Dict[SampleKey, Dict] = {}

for key, slot_map in samples.items():
    missing = [s for s in REQUIRED_SLOTS if slot_map[s] is None]
    if missing:
        incomplete_samples[key] = {'missing': missing, **slot_map}
        logger.warning('Incomplete (%s, %s) missing: %s', *key, missing)
    else:
        complete_samples[key] = slot_map

logger.info('Complete  : %d', len(complete_samples))
logger.info('Incomplete: %d', len(incomplete_samples))
print(f'Complete   samples: {len(complete_samples):,}')
print(f'Incomplete samples: {len(incomplete_samples):,}')

if incomplete_samples:
    _df = pd.DataFrame(
        [(d, u, v['missing']) for (d, u), v in incomplete_samples.items()],
        columns=['disaster', 'uid', 'missing_slots'],
    )
    inc_out = LOG_DIR / 'incomplete_samples.csv'
    _df.to_csv(inc_out, index=False)
    print(f'Report saved -> {inc_out}')
    print(_df.head(10).to_string(index=False))

2026-04-25 22:41:28  INFO      Complete  : 2799
2026-04-25 22:41:28  INFO      Incomplete: 0


Complete   samples: 2,799
Incomplete samples: 0


## Label JSON Parsing

### Parse all label JSONs

In [17]:
all_buildings: List[Dict[str, Any]] = []
empty_count = missing_count = 0

sample_items = list(complete_samples.items()) or list(samples.items())

for (disaster, uid), slot_map in tqdm(sample_items, desc='Parsing labels'):
    label_path = slot_map.get('label')
    if label_path is None:
        missing_count += 1
        continue
    buildings = parse_label_json(label_path)
    if not buildings:
        empty_count += 1
        continue
    for b in buildings:
        all_buildings.append({
            'disaster':     disaster,
            'scene_uid':    uid,
            'building_uid': b['uid'],
            'feature_type': b['feature_type'],
            'damage_type':  b['damage_type'],
            'lng_lat_wkt':  b['lng_lat_wkt'],  # WKT string (geographic coords)
            'xy_wkt':       b['xy_wkt'],        # WKT string (pixel coords)
        })

df_labels = pd.DataFrame(all_buildings) if all_buildings else pd.DataFrame()
print(f'Building records : {len(df_labels):,}')
print(f'Empty label files: {empty_count}')
print(f'Missing labels   : {missing_count}')

Parsing labels: 100%|██████████| 2799/2799 [00:01<00:00, 1489.31it/s]


Building records : 162,787
Empty label files: 516
Missing labels   : 0


### Label statistics

In [18]:
if not df_labels.empty:
    print('-- Damage distribution --')
    print(df_labels['damage_type'].value_counts().to_string())
    print('\n-- Buildings per disaster --')
    print(df_labels.groupby('disaster')['building_uid'].count().to_string())

    labels_out = PROCESSED_DIR / 'buildings_annotations.csv'
    labels_out.parent.mkdir(parents=True, exist_ok=True)
    df_labels.to_csv(labels_out, index=False)
    print(f'\nSaved -> {labels_out}')
    df_labels.head()

-- Damage distribution --
damage_type
un-classified    162787

-- Buildings per disaster --
disaster
guatemala-volcano        856
hurricane-florence      6446
hurricane-harvey       23014
hurricane-matthew      13939
hurricane-michael      22686
mexico-earthquake      32271
midwest-flooding        8756
palu-tsunami           31394
santa-rosa-wildfire    12950
socal-fire             10475

Saved -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/processed/buildings_annotations.csv


## Dataset Restructuring

Organises files into `processed/train/<disaster>/<uid>/` containing `pre.png`, `post.png`, `target.png`, `label.json`.

In [19]:
COPY_FILES = False

_SLOT_TO_DEST: Dict[str, str] = {
    'pre': 'pre.png', 'post': 'post.png',
    'target': 'target.png', 'label': 'label.json',
}

def _link_or_copy(src: Path, dst: Path, *, copy: bool = False) -> None:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst) if copy else dst.symlink_to(src.resolve())

def restructure_dataset(
    all_samples: Dict[SampleKey, Dict],
    out_root: Path,
    *,
    copy_files: bool = False,
    skip_incomplete: bool = True,
) -> Dict[str, Any]:
    out_root.mkdir(parents=True, exist_ok=True)
    stats: Dict[str, Any] = {
        'total': len(all_samples), 'processed': 0,
        'skipped': 0, 'errors': 0,
        'per_disaster': defaultdict(int),
    }
    for (disaster, uid), slot_map in tqdm(
        all_samples.items(), desc='Restructuring', unit='sample'
    ):
        if skip_incomplete and any(slot_map.get(s) is None for s in REQUIRED_SLOTS):
            stats['skipped'] += 1
            continue
        sample_dir = out_root / disaster / uid
        sample_dir.mkdir(parents=True, exist_ok=True)
        try:
            for slot, dest_name in _SLOT_TO_DEST.items():
                src = slot_map.get(slot)
                if src is not None:
                    _link_or_copy(src, sample_dir / dest_name, copy=copy_files)
            stats['processed'] += 1
            stats['per_disaster'][disaster] += 1
        except Exception as exc:
            logger.error('Error (%s, %s): %s', disaster, uid, exc)
            stats['errors'] += 1
    return stats

restructure_stats = restructure_dataset(
    samples, PROCESSED_TRAIN_DIR,
    copy_files=COPY_FILES, skip_incomplete=True,
)
print('-- Restructuring Stats --')
for k, v in restructure_stats.items():
    if k != 'per_disaster':
        print(f'  {k:12s}: {v:,}')
print('\n-- Per-disaster --')
pd.Series(dict(restructure_stats['per_disaster']), name='samples').sort_values(ascending=False)

Restructuring: 100%|██████████| 2799/2799 [00:02<00:00, 1294.20sample/s]


-- Restructuring Stats --
  total       : 2,799
  processed   : 2,799
  skipped     : 0
  errors      : 0

-- Per-disaster --


socal-fire             823
hurricane-michael      343
hurricane-harvey       319
hurricane-florence     319
midwest-flooding       279
hurricane-matthew      238
santa-rosa-wildfire    226
mexico-earthquake      121
palu-tsunami           113
guatemala-volcano       18
Name: samples, dtype: int64

## Validation & Final Report

### Validate processed directory

In [20]:
EXPECTED = {'pre.png', 'post.png', 'target.png', 'label.json'}
validation_errors: List[Dict] = []
validated_count = 0

for ddir in sorted(PROCESSED_TRAIN_DIR.glob('*/')):
    for udir in sorted(ddir.glob('*/')):
        actual  = {p.name for p in udir.iterdir()}
        missing = EXPECTED - actual
        if missing:
            validation_errors.append({
                'disaster': ddir.name,
                'uid': udir.name,
                'missing': sorted(missing),
            })
        else:
            validated_count += 1

print(f'Valid samples   : {validated_count:,}')
print(f'Validation errors: {len(validation_errors)}')

if validation_errors:
    df_err = pd.DataFrame(validation_errors)
    err_out = LOG_DIR / 'validation_errors.csv'
    df_err.to_csv(err_out, index=False)
    print(f'Errors saved -> {err_out}')
    print(df_err.head(10).to_string(index=False))

Valid samples   : 2,799
Validation errors: 0


### Generate train manifest

In [21]:
manifest_rows: List[Dict] = []
for ddir in sorted(PROCESSED_TRAIN_DIR.glob('*/')):
    for udir in sorted(ddir.glob('*/')):
        manifest_rows.append({
            'disaster':    ddir.name,
            'uid':         udir.name,
            'sample_dir':  str(udir),
            'pre_path':    str(udir / 'pre.png')    if (udir / 'pre.png').exists()    else '',
            'post_path':   str(udir / 'post.png')   if (udir / 'post.png').exists()   else '',
            'target_path': str(udir / 'target.png') if (udir / 'target.png').exists() else '',
            'label_path':  str(udir / 'label.json') if (udir / 'label.json').exists() else '',
        })

df_manifest = pd.DataFrame(manifest_rows)
manifest_path = PROCESSED_DIR / 'train_manifest.csv'
df_manifest.to_csv(manifest_path, index=False)
print(f'Manifest saved -> {manifest_path}  ({len(df_manifest):,} rows)')
df_manifest.head()

Manifest saved -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/processed/train_manifest.csv  (2,799 rows)


,disaster,uid,sample_dir,pre_path,post_path,target_path,label_path
0,guatemala-volcano,00000000,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...
1,guatemala-volcano,00000001,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...
2,guatemala-volcano,00000002,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...
3,guatemala-volcano,00000006,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...
4,guatemala-volcano,00000007,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...,/home/prosper/Desktop/Python/Projects/Project_...


### Final summary

In [22]:
SEP = '=' * 60
lines = [
    SEP,
    '  ResQVision - Data Collection Pipeline  COMPLETE',
    SEP,
    f'  Raw images          : {n_images:>8,}',
    f'  Raw targets         : {n_targets:>8,}',
    f'  Raw labels          : {n_labels:>8,}',
    '',
    f'  Unique scene pairs  : {len(samples):>8,}',
    f'  Complete pairs      : {len(complete_samples):>8,}',
    f'  Incomplete pairs    : {len(incomplete_samples):>8,}',
    '',
    f'  Samples processed   : {restructure_stats["processed"]:>8,}',
    f'  Samples skipped     : {restructure_stats["skipped"]:>8,}',
    '',
    f'  Validated samples   : {validated_count:>8,}',
    f'  Validation errors   : {len(validation_errors):>8,}',
    '',
    f'  Building annotations: {len(df_labels):>8,}',
    SEP,
]
report = '\n'.join(lines)
print(report)
logger.info('\n%s', report)

report_path = LOG_DIR / 'ingestion_summary.txt'
report_path.write_text(report, encoding='utf-8')
print(f'\nReport saved -> {report_path}')

2026-04-25 22:41:39  INFO      
  ResQVision - Data Collection Pipeline  COMPLETE
  Raw images          :    5,598
  Raw targets         :    5,598
  Raw labels          :    5,598

  Unique scene pairs  :    2,799
  Complete pairs      :    2,799
  Incomplete pairs    :        0

  Samples processed   :    2,799
  Samples skipped     :        0

  Validated samples   :    2,799
  Validation errors   :        0

  Building annotations:  162,787


  ResQVision - Data Collection Pipeline  COMPLETE
  Raw images          :    5,598
  Raw targets         :    5,598
  Raw labels          :    5,598

  Unique scene pairs  :    2,799
  Complete pairs      :    2,799
  Incomplete pairs    :        0

  Samples processed   :    2,799
  Samples skipped     :        0

  Validated samples   :    2,799
  Validation errors   :        0

  Building annotations:  162,787

Report saved -> /home/prosper/Desktop/Python/Projects/Project_019_ResQVision/service/data/logs/ingestion_summary.txt
